In [11]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
from figure_formatting import setup_figure, save_figure
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

In [2]:
def clean_metrics_name(name):
    name = name.replace("_", " ")
    name = re.sub(r"\s*mm[23]?\s*$", "", name, flags=re.IGNORECASE)
    name = name.strip()
    if name.upper().startswith("NODDI") or name.upper().startswith("DKI"): return " ".join([p.upper() for p in name.split()])
    ordinal_allowed = ("quarter" in name.lower())
    def ordinalize(s):
        s = s.lower()
        if s.isdigit():
            n = int(s)
            if n % 10 == 1: return f"{n}st"
            elif n % 10 == 2: return f"{n}nd"
            elif n % 10 == 3: return f"{n}rd"
            else: return f"{n}th"
        return s
    stopwords = {"of", "and", "in", "for", "to", "with", "on", "at"}
    cleaned = []
    for w in name.split():
        if w.isdigit(): cleaned.append(ordinalize(w) if ordinal_allowed else w)
        elif w.lower() in stopwords: cleaned.append(w.lower())
        else: cleaned.append(w.capitalize())
    return " ".join(cleaned)

In [6]:
ROOT_DIR = Path("/Users/bmacedo/Desktop/final_WM")
INPUT_DIR = ROOT_DIR / "input_data"

abbrevs = pd.read_excel(INPUT_DIR / "abbreviations.xlsx", sheet_name="Sheet1")
annots = pd.read_csv(INPUT_DIR / "tract_sa_axis_ranges_thresh50.csv")

tracts = ['AssociationArcuateFasciculusL','AssociationArcuateFasciculusR', 'AssociationCingulumL', 'AssociationCingulumR',
          'AssociationExtremeCapsuleL', 'AssociationExtremeCapsuleR', 'AssociationFrontalAslantTractL','AssociationFrontalAslantTractR',
          'AssociationHippocampusAlveusL', 'AssociationHippocampusAlveusR', 'AssociationInferiorFrontoOccipitalFasciculusL',
          'AssociationInferiorFrontoOccipitalFasciculusR','AssociationInferiorLongitudinalFasciculusL',
          'AssociationInferiorLongitudinalFasciculusR','AssociationMiddleLongitudinalFasciculusL', 'AssociationMiddleLongitudinalFasciculusR',
          'AssociationParietalAslantTractL','AssociationParietalAslantTractR','AssociationSuperiorLongitudinalFasciculusL',
          'AssociationSuperiorLongitudinalFasciculusR', 'AssociationUncinateFasciculusL', 'AssociationUncinateFasciculusR',
          'AssociationVerticalOccipitalFasciculusL', 'AssociationVerticalOccipitalFasciculusR','CerebellumCerebellumL',
          'CerebellumCerebellumR','CerebellumInferiorCerebellarPeduncleL','CerebellumInferiorCerebellarPeduncleR',
          'CerebellumMiddleCerebellarPeduncle', 'CerebellumSuperiorCerebellarPeduncle','CerebellumVermis', 'CommissureCorpusCallosum',
          'ProjectionBasalGangliaAcousticRadiationL', 'ProjectionBasalGangliaAcousticRadiationR','ProjectionBasalGangliaAnsaLenticularisL',
          'ProjectionBasalGangliaAnsaLenticularisR', 'ProjectionBasalGangliaAnsaSubthalamicaL', 'ProjectionBasalGangliaAnsaSubthalamicaR',
          'ProjectionBasalGangliaCorticostriatalTractL', 'ProjectionBasalGangliaCorticostriatalTractR', 
          'ProjectionBasalGangliaFasciculusLenticularisL','ProjectionBasalGangliaFasciculusLenticularisR',
          'ProjectionBasalGangliaFasciculusSubthalamicusL', 'ProjectionBasalGangliaFasciculusSubthalamicusR', 'ProjectionBasalGangliaFornixL',
          'ProjectionBasalGangliaFornixR','ProjectionBasalGangliaOpticRadiationL', 'ProjectionBasalGangliaOpticRadiationR',
          'ProjectionBasalGangliaThalamicRadiationL', 'ProjectionBasalGangliaThalamicRadiationR', 'ProjectionBrainstemCorticopontineTractL',
          'ProjectionBrainstemCorticopontineTractR', 'ProjectionBrainstemCorticospinalTractL', 'ProjectionBrainstemCorticospinalTractR',
          'ProjectionBrainstemMedialForebrainBundleL', 'ProjectionBrainstemMedialForebrainBundleR', 'ProjectionBrainstemMedialLemniscusL',
          'ProjectionBrainstemMedialLemniscusR', 'ProjectionBrainstemNonDecussatingDentatorubrothalamicTractL',
          'ProjectionBrainstemNonDecussatingDentatorubrothalamicTractR', 'ProjectionBrainstemReticularTractL', 
          'ProjectionBrainstemReticularTractR']

In [7]:
abbrevs.head(2)

,Tract,Abbreviation,HCPYA_1065,Hemisphere,Full_Name,Full_Name_capitalized,Tract_Long_Name,new_qsirecon_tract_names,Other_Nomenclatures,Type
0,AF_left,AF,AF_L,Left,arcuate fasciculus,Left Arcuate Fasciculus,Arcuate_Fasciculus_L,Association_ArcuateFasciculusL,NaN,Association
1,C_FPH_left,C_FPH,C_FPH_L,Left,"cingulum, frontal parahippocampal segment","Left Cingulum, Frontal Parahippocampal Segment",Cingulum_Frontal_Parahippocampal_L,NaN,NaN,Association


In [8]:
annots.head(2)

,Tract,SA_Range,N_Regions,Hemisphere,Abbreviation
0,FAT_left,168.0,26,left,FAT
1,AF_left,173.0,46,left,AF


In [9]:
def clean_name(name): return str(name).replace("_", "").lower() if pd.notna(name) else None

abbrevs["clean_name"] = abbrevs["new_qsirecon_tract_names"].apply(clean_name)

tracts_df = pd.DataFrame({"new_qsirecon_tract_names": tracts, "clean_name": [clean_name(t) for t in tracts]})

abbrev_clean_list = abbrevs["clean_name"].dropna().tolist()
missing_tracts = [t for t in tracts if clean_name(t) not in abbrev_clean_list]

print("Tracts not found in abbrevs['new_qsirecon_tract_names'] (after cleaning):")
print(missing_tracts)
print(f"Total missing: {len(missing_tracts)}\n")

# merge to get tract and s-a range
tracts_with_sa = (tracts_df.merge(abbrevs[["clean_name", "Tract"]], on="clean_name", how="left")
                  .merge(annots[["Tract", "SA_Range"]], on="Tract", how="left").drop(columns=["clean_name"]))

tracts_with_sa_clean = tracts_with_sa.dropna()

for _, row in tracts_with_sa_clean.iterrows():
    print(row.to_dict())

Tracts not found in abbrevs['new_qsirecon_tract_names'] (after cleaning):
['AssociationExtremeCapsuleL', 'AssociationExtremeCapsuleR', 'AssociationHippocampusAlveusL', 'AssociationHippocampusAlveusR', 'CerebellumCerebellumL', 'CerebellumCerebellumR', 'CerebellumInferiorCerebellarPeduncleL', 'CerebellumInferiorCerebellarPeduncleR', 'CerebellumMiddleCerebellarPeduncle', 'CerebellumSuperiorCerebellarPeduncle', 'CerebellumVermis', 'CommissureCorpusCallosum', 'ProjectionBasalGangliaAcousticRadiationL', 'ProjectionBasalGangliaAcousticRadiationR', 'ProjectionBasalGangliaAnsaLenticularisL', 'ProjectionBasalGangliaAnsaLenticularisR', 'ProjectionBasalGangliaAnsaSubthalamicaL', 'ProjectionBasalGangliaAnsaSubthalamicaR', 'ProjectionBasalGangliaFasciculusLenticularisL', 'ProjectionBasalGangliaFasciculusLenticularisR', 'ProjectionBasalGangliaFasciculusSubthalamicusL', 'ProjectionBasalGangliaFasciculusSubthalamicusR', 'ProjectionBrainstemCorticopontineTractL', 'ProjectionBrainstemCorticopontineTractR

In [14]:
# dirs
ROOT_DIR = Path("/Users/bmacedo/Desktop/final_WM")
FIG3_DIR = ROOT_DIR / "output_data" / "results" / "main_figures" / "figure3"
SUPPFIG4_DIR = ROOT_DIR / "output_data" / "results" / "supplemental_figures" / "suppfig4"
for d in [FIG3_DIR, SUPPFIG4_DIR]: d.mkdir(parents=True, exist_ok=True)

scenarios = [("NODDI", False), ("DKI", False)]

for scenario_name, scenario_etiv in scenarios:
    print(f"\n[SCENARIO] {scenario_name} | eTIV={scenario_etiv}")

    if scenario_name == "NODDI" and scenario_etiv is False:
        save_dir, out_base, file_stub = FIG3_DIR, "pca_ALL_LRkept", "PC1_loading_and_scatter"
    elif scenario_name == "DKI" and scenario_etiv is False:
        save_dir, out_base, file_stub = SUPPFIG4_DIR, "pca_ALL_LRkept_dki", "PC1_loading_and_scatter_dki"

    pc_scores_df = pd.read_csv(save_dir / f"{out_base}_bundle_scores_PC12.csv")
    pc_loadings_df = pd.read_csv(save_dir / f"{out_base}_metric_loadings_PC12.csv")

    pc1_loadings = pc_loadings_df.set_index("metric")["PC1"].dropna().sort_values(ascending=True)
    pc1_loadings.index = [clean_metrics_name(m) for m in pc1_loadings.index]
    if pc1_loadings.index.duplicated().any():
        counts = {}
        new_idx = []
        for lbl in pc1_loadings.index:
            counts[lbl] = counts.get(lbl, 0) + 1
            new_idx.append(lbl if counts[lbl] == 1 else f"{lbl} ({counts[lbl]})")
        pc1_loadings.index = new_idx

    df_merged = pc_scores_df.merge(tracts_with_sa_clean[["new_qsirecon_tract_names", "SA_Range"]],
                                   left_on="bundle", right_on="new_qsirecon_tract_names", how="inner")
    data_scatter = df_merged[["SA_Range", "PC1"]].dropna()

    x = data_scatter["SA_Range"].values
    y = data_scatter["PC1"].values
    r, _ = pearsonr(x, y)

    n_perm = 10000
    r_perm = np.empty(n_perm)
    rng = np.random.default_rng(0)
    for i in range(n_perm):
        x_perm = rng.permutation(x)
        r_perm[i], _ = pearsonr(x_perm, y)

    p_perm = (np.sum(np.abs(r_perm) >= np.abs(r)) + 1) / (n_perm + 1)

    sns.set_style("white")
    fig, axes = setup_figure(width_mm=150, height_mm=60, margins_mm=(42, 10, 10, 2),
                             axes_linewidth=0.8, nrows=1, ncols=2, sharex=False, sharey=False)
    ax0, ax1 = axes.ravel()
    fig.subplots_adjust(wspace=0.35)

    ax0.barh(pc1_loadings.index, pc1_loadings.values, color="#7BA6E6")
    ax0.axvline(0, color="black", lw=1)
    ax0.set_xlabel("PC1 Loadings")
    ax0.set_ylabel("")
    sns.despine(ax=ax0, top=True, right=True)

    sns.regplot(data=data_scatter, x="SA_Range", y="PC1", scatter=False, ci=95,
                line_kws={"color": "black", "lw": 2}, ax=ax1)
    sc = ax1.scatter(data_scatter["SA_Range"], data_scatter["PC1"], c=data_scatter["PC1"],
                     cmap="viridis", s=45, alpha=0.85, edgecolor="none")

    ax1.set_xlabel("SA Range")
    ax1.set_ylabel("PC1 Score")
    sns.despine(ax=ax1, top=True, right=True)

    ax1.text(0.05, 0.92,
             rf"$\it{{r}}$ = {r:.2f}" + "\n" + rf"$\it{{p}}_{{perm}}$ = {p_perm:.3f}",
             transform=ax1.transAxes, fontsize=7, ha="left", va="top")

    ax0.tick_params(axis="both", which="both", bottom=True, left=True)
    ax1.tick_params(axis="both", which="both", bottom=True, left=True)

    cax = inset_axes(ax1, width="3%", height="30%", loc="upper left",
                     bbox_to_anchor=(1.02, 0, 1, 1), bbox_transform=ax1.transAxes, borderpad=0)
    cbar = fig.colorbar(sc, cax=cax)
    cbar.set_label("PC1 score")
    cbar.set_ticks([-6, 0, 6])
    cbar.set_ticklabels(["-6", "0", "6"])
    cbar.ax.tick_params(which="both", length=0)

    svg_path = save_dir / f"{file_stub}.svg"
    png_path = save_dir / f"{file_stub}.png"
    save_figure(fig, svg_path, autofit=False)
    fig.savefig(png_path, dpi=600, bbox_inches="tight")

    print(f"[SAVE] → {svg_path}")
    print(f"[SAVE] → {png_path}")
    plt.close(fig)


[SCENARIO] NODDI | eTIV=False
[SAVE] → /Users/bmacedo/Desktop/final_WM/output_data/results/main_figures/figure3/PC1_loading_and_scatter.svg
[SAVE] → /Users/bmacedo/Desktop/final_WM/output_data/results/main_figures/figure3/PC1_loading_and_scatter.png

[SCENARIO] DKI | eTIV=False
[SAVE] → /Users/bmacedo/Desktop/final_WM/output_data/results/supplemental_figures/suppfig4/PC1_loading_and_scatter_dki.svg
[SAVE] → /Users/bmacedo/Desktop/final_WM/output_data/results/supplemental_figures/suppfig4/PC1_loading_and_scatter_dki.png
